Ma ugyanaz a kérdés három szinten: **chat**, **munkafolyamat**, **ágens** (ma csak a terve). Az Ollama fusson, mindkét modellel (`gemma4:e4b`, `bge-m3`), 8192 tokenes kontextussal (lásd a telepítési oldalt). Futtasd sorban: **Shift + Enter**. A Copilot kódját olvasd el, mielőtt **Keep**-et nyomsz; ha rossz, **Undo**.

In [ ]:
import json
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pymupdf

import kurzus
from kurzus import chat, embed, kerdez, utolso_hasznalat
from kurzus import ellenoriz, ellenoriz_feltetel, ellenoriz_fuggveny

## 1. Előkészítés: indítsd el most

6 jelentés (a múltkori öt + a 2023. szeptemberi, a mintaprojekt miatt), a 6. alkalom javított darabolásával, saját gyorsítótárba. **Futtasd le most a négy cellát**: a beágyazás nálam kb. 2-2,5 perc, a checkpoint alatt elkészül.

In [ ]:
# A 3. alkalom függvényei: PDF → bekezdések → darabok. Csak futtasd.
FEJLEC_JELEK = ("INFLÁCIÓS JELENTÉS •", "MAGYAR NEMZETI BANK")
DARABOK_FAJL = Path("data/cache/darabok.json")


def tisztit(szoveg):
    """PDF-ből kinyert szöveg takarítása: elválasztás, pontsorok, sortörések, szóközök."""
    szoveg = re.sub(r"-\n(?=[a-záéíóöőúüű])", "", szoveg)  # sor végi elválasztás: ren-|delkezésére
    szoveg = re.sub(r"\.{4,}", " ", szoveg)                # tartalomjegyzék pontsora
    szoveg = szoveg.replace("\n", " ")
    return re.sub(r"\s+", " ", szoveg).strip()


def nyomtatott_oldalszam(oldal):
    """Az élőfejben álló nyomtatott oldalszám; ha az oldalon nincs élőfej (borító, belső borító), None."""
    elso_sorok = [s.strip() for s in oldal.get_text().splitlines()[:5]]
    if not any("INFLÁCIÓS JELENTÉS •" in s for s in elso_sorok):
        return None
    for sor in elso_sorok:
        if sor.isdigit():
            return int(sor)
    return None


def oldal_bekezdesei(oldal):
    """Az oldal bekezdései takarítva, élőfej, grafikonfeliratok és oldalszám nélkül."""
    bekezdesek = []
    for blokk in oldal.get_text("blocks"):
        szoveg = tisztit(blokk[4])
        if any(jel in szoveg for jel in FEJLEC_JELEK) or szoveg.isupper():
            continue
        if len(szoveg.split()) < 5 or not re.search(r"[a-zA-ZáéíóöőúüűÁÉÍÓÖŐÚÜŰ]{3}", szoveg):
            continue
        bekezdesek.append(szoveg)
    return bekezdesek


def jelentes_oldalai(pdf_utvonal, jelentes):
    """Egy jelentés oldalai: jelentés, PDF-oldalszám, nyomtatott oldalszám és a bekezdések."""
    return [{"jelentes": jelentes, "pdf_oldal": i, "oldal": nyomtatott_oldalszam(o), "bekezdesek": oldal_bekezdesei(o)}
            for i, o in enumerate(pymupdf.open(pdf_utvonal), start=1)]


def darabol_fix(szoveg, meret=1000, atfedes=0):
    """Fix méretű darabok (karakterben); az egymást követő darabok `atfedes` karakternyit átfednek."""
    lepes = meret - atfedes
    return [szoveg[i:i + meret] for i in range(0, max(len(szoveg) - atfedes, 1), lepes)]


ROVIDITESEK = {"pl", "kb", "stb", "ill", "ún", "vö", "szp", "Kr", "dr"}


def mondatokra(szoveg):
    """Mondatokra bontja a szöveget. Mondatvég: pont, kérdőjel vagy felkiáltójel, utána szóköz és
    nagybetű vagy számjegy; kivéve, ha a pont előtt szám, római szám, egyetlen betű vagy rövidítés áll."""
    mondatok = []
    for resz in re.split(r"(?<=[.!?]) (?=[A-ZÁÉÍÓÖŐÚÜŰ0-9])", szoveg):
        elozo_szo = mondatok[-1].split()[-1].rstrip(".") if mondatok else ""
        if elozo_szo.isdigit() or re.fullmatch(r"[IVX]+|\w", elozo_szo) or elozo_szo in ROVIDITESEK:
            mondatok[-1] += " " + resz
        else:
            mondatok.append(resz)
    return mondatok


def darabol_bekezdesenkent(bekezdesek, max_hossz=1000):
    """Egész bekezdéseket fűz össze, amíg a darab el nem érné a `max_hossz` karaktert. A túl hosszú
    bekezdést mondatokra bontja; ha egyetlen mondat is túl hosszú, azt szavakra."""
    reszek = []
    for bekezdes in bekezdesek:
        if len(bekezdes) <= max_hossz:
            reszek.append(bekezdes)
            continue
        for mondat in mondatokra(bekezdes):
            reszek += [mondat] if len(mondat) <= max_hossz else mondat.split()
    darabok, aktualis = [], ""
    for resz in reszek:
        if aktualis and len(aktualis) + 1 + len(resz) > max_hossz:
            darabok.append(aktualis)
            aktualis = resz
        else:
            aktualis = (aktualis + " " + resz).strip()
    if aktualis:
        darabok.append(aktualis)
    return darabok


def darabok_keszitese(oldalak, max_hossz=1000):
    """Egy jelentés darabjai; minden darab megőrzi a jelentést és mindkét oldalszámot."""
    darabok = []
    for o in oldalak:
        for j, szoveg in enumerate(darabol_bekezdesenkent(o["bekezdesek"], max_hossz)):
            darabok.append({"id": f'{o["jelentes"]}_p{o["pdf_oldal"]:03d}_{j}', "jelentes": o["jelentes"],
                            "pdf_oldal": o["pdf_oldal"], "oldal": o["oldal"], "szoveg": szoveg})
    return darabok


def darabok_betoltese(fajl=DARABOK_FAJL):
    """A forrasok.csv összes jelentésének darabjai a gyorsítótárból; ha nincs meg, újra elkészíti és elmenti (kb. 10 mp)."""
    if fajl.exists():
        return json.loads(fajl.read_text(encoding="utf-8"))
    print("Nincs meg a darabok.json: újra elkészítem a 3. alkalom függvényeivel.")
    darabok = []
    for s in pd.read_csv("data/mnb/forrasok.csv", sep=";").itertuples():
        darabok += darabok_keszitese(jelentes_oldalai(kurzus.letolt(s.url, "data/mnb/" + s.fajl), s.jelentes))
    fajl.parent.mkdir(parents=True, exist_ok=True)
    fajl.write_text(json.dumps(darabok, ensure_ascii=False), encoding="utf-8")
    return darabok

In [ ]:
# A 4. alkalom függvényei: beágyazás, index, keresés, ellenőrzés a nyers PDF-ből. Csak futtasd.
INDEX_FAJL = Path("data/cache/embeddingek_rag.npz")    # saját fájl: más jelentések, mint a 4. alkalommal


def beagyaz(szovegek, adag=64):
    """Szövegek embeddingje adagokban (egy hívás = `adag` szöveg); numpy-mátrixot ad vissza."""
    vektorok = []
    for i in range(0, len(szovegek), adag):
        vektorok += embed(szovegek[i:i + adag])
    return np.array(vektorok, dtype=np.float32)


def index_betoltese(darabok, fajl=INDEX_FAJL):
    """A darabok normalizált embedding-mátrixa: a gyorsítótárból, ha ugyanezekhez a darabokhoz
    készült, különben kiszámolja és elmenti. Az i. sor az i. darabé."""
    idk = [d["id"] for d in darabok]
    if fajl.exists():
        tarolt = np.load(fajl)
        if list(tarolt["idk"]) == idk:
            print(f"Gyorsítótárból: {len(idk)} darab (a kiszámolás {float(tarolt['ido']):.0f} mp volt).")
            return tarolt["vektorok"]
    kezdes = time.perf_counter()
    vektorok = beagyaz([d["szoveg"] for d in darabok])
    vektorok = vektorok / np.linalg.norm(vektorok, axis=1, keepdims=True)   # normalizálás
    ido = time.perf_counter() - kezdes
    fajl.parent.mkdir(parents=True, exist_ok=True)
    np.savez(fajl, vektorok=vektorok, idk=np.array(idk), ido=ido)
    print(f"Kiszámolva: {len(idk)} darab, {ido:.0f} mp ({len(idk) / ido:.1f} darab/mp).")
    return vektorok


def keres(kerdes, darabok, index, k=5):
    """A kérdéshez leghasonlóbb k darab, (hasonlóság, darab) párokként, csökkenő sorrendben."""
    q = np.array(embed(kerdes)[0])
    q = q / np.linalg.norm(q)
    hasonlosagok = index @ q                       # koszinusz-hasonlóság az összes darabbal egyszerre
    legjobbak = np.argsort(-hasonlosagok)[:k]      # a k legnagyobb hasonlóság helye
    return [(float(hasonlosagok[i]), darabok[i]) for i in legjobbak]


def kiir(talalatok):
    """A találatok kiírása: hasonlóság, forrás, a szöveg eleje."""
    for hasonlosag, d in talalatok:
        hol = f'{d["oldal"]}. oldal' if d["oldal"] else f'PDF {d["pdf_oldal"]}. oldal'
        print(f'{hasonlosag:.3f}  {d["jelentes"]}, {hol}:  {d["szoveg"][:140]}…')


forrasok = pd.read_csv("data/mnb/forrasok.csv", sep=";")


def oldalak_ahol(minta, jelentesek):
    """(jelentés, PDF-oldal) párok, ahol a nyers PDF-szövegben megvan a minta (reguláris kifejezés)."""
    talalt = set()
    for s in forrasok[forrasok["jelentes"].isin(jelentesek)].itertuples():
        for i, oldal in enumerate(pymupdf.open("data/mnb/" + s.fajl), start=1):
            if re.search(minta, tisztit(oldal.get_text())):
                talalt.add((s.jelentes, i))
    return talalt


def van_jo_talalat(talalatok, vart_oldalak):
    """Igaz, ha a találatok között van olyan darab, amely a várt oldalak egyikéről jön."""
    return any((d["jelentes"], d["pdf_oldal"]) in vart_oldalak for _, d in talalatok)

In [ ]:
# Az 5. alkalom függvényei: a RAG-lánc és az ellenőrzései. Csak futtasd.
HONAPOK = {"01": "január", "02": "február", "03": "március", "04": "április", "05": "május",
           "06": "június", "07": "július", "08": "augusztus", "09": "szeptember", "10": "október",
           "11": "november", "12": "december"}


def forras_cimke(d):
    """Forrásmegjelölés egy darabhoz, például: MNB Inflációs jelentés, 2025. március, 62. oldal."""
    ev, honap = d["jelentes"].split("-")
    hol = f'{d["oldal"]}. oldal' if d["oldal"] else f'PDF {d["pdf_oldal"]}. oldal'
    return f"MNB Inflációs jelentés, {ev}. {HONAPOK[honap]}, {hol}"


def kontextus(talalatok):
    """A modellnek szánt források: sorszám, forrásmegjelölés, aztán a darab szövege."""
    reszek = []
    for i, (hasonlosag, d) in enumerate(talalatok, start=1):
        reszek.append(f"[{i}] {forras_cimke(d)}\n{d['szoveg']}")
    return "\n\n".join(reszek)


def tokenszam(uzenetek):
    """Hány token a kérés a helyi modell szerint. Szöveget vagy üzenetlistát is elfogad.
    Egyetlen tokennyi választ kérünk, gondolkodás nélkül, hogy gyors legyen (1-2 mp)."""
    if isinstance(uzenetek, str):
        uzenetek = [{"role": "user", "content": uzenetek}]
    chat(uzenetek, gondolkodas="none", max_tokens=1)
    return utolso_hasznalat["prompt_tokens"]


RENDSZER = """Az MNB Inflációs jelentéseiből válaszolsz közgazdász hallgatóknak, magyarul, röviden.
Szabályok:
1. Csak a megadott forrásokból válaszolj. Saját tudásból ne írj számot.
2. Minden állítás után szögletes zárójelben jelöld a forrás számát, például [2].
3. Figyelj a jelentés dátumára: a régebbi jelentés előrejelzése nem tényadat. Ha a források eltérnek, a legújabb jelentésből vett tényadatot add meg.
4. Ha a források nem tartalmazzák a választ, ezt írd: "A forrásokban nincs benne." Ilyenkor ne írj számot."""


TOKEN_HATAR = 3000   # a 4096 tokenes ablakból ennyi a kérés; a többi a gondolkodásé és a válaszé


def rag_valasz(kerdes, darabok, index, k=5, rendszer=RENDSZER, gondolkodas="none"):
    """A teljes RAG-lánc: keresés → kontextus → tokenszám → modell. Szótárat ad vissza."""
    talalatok = keres(kerdes, darabok, index, k)
    while True:
        uzenetek = [{"role": "system", "content": rendszer},
                    {"role": "user", "content": f"Források:\n\n{kontextus(talalatok)}\n\nKérdés: {kerdes}"}]
        tokenek = tokenszam(uzenetek)
        if tokenek <= TOKEN_HATAR or len(talalatok) == 1:
            break
        talalatok = talalatok[:-1]           # nem fér bele: a leggyengébb találat kimarad
    print(f"A kérés: {tokenek} token, {len(talalatok)} darab (határ: {TOKEN_HATAR}, ablak: 4096)")
    kezdes = time.perf_counter()
    valasz = chat(uzenetek, gondolkodas=gondolkodas)["content"]
    return {"kerdes": kerdes, "valasz": valasz, "talalatok": talalatok, "tokenek": tokenek,
            "valasz_tokenek": utolso_hasznalat.get("completion_tokens"),
            "ido": time.perf_counter() - kezdes}


def hivatkozasok(valasz):
    """A válaszban szögletes zárójelben álló forrásszámok, növekvő sorrendben, ismétlés nélkül."""
    szamok = set()
    for belseje in re.findall(r"\[([\d,\s]+)\]", valasz):     # [2] vagy [2, 3]
        szamok.update(int(x) for x in re.findall(r"\d+", belseje))
    return sorted(szamok)


def kiir_valasz(eredmeny):
    """A válasz, alatta a hivatkozott források teljes megjelölésével (jelentés, nyomtatott oldal)."""
    print(eredmeny["valasz"])
    print("\nHivatkozott források:")
    for n in hivatkozasok(eredmeny["valasz"]):
        if 1 <= n <= len(eredmeny["talalatok"]):
            print(f"  [{n}] {forras_cimke(eredmeny['talalatok'][n - 1][1])}")
        else:
            print(f"  [{n}] nincs ilyen sorszámú forrás: a modell kitalálta")


_oldal_tar = {}


def oldal_szovege(jelentes, pdf_oldal):
    """Egy PDF-oldal nyers szövege, takarítva. A darabolástól független: ezzel ellenőrzünk."""
    if (jelentes, pdf_oldal) not in _oldal_tar:
        fajl = forrasok.loc[forrasok["jelentes"] == jelentes, "fajl"].iloc[0]
        _oldal_tar[(jelentes, pdf_oldal)] = tisztit(pymupdf.open("data/mnb/" + fajl)[pdf_oldal - 1].get_text())
    return _oldal_tar[(jelentes, pdf_oldal)]


def szam_a_pdfbol(minta, jelentes):
    """A minta zárójeles részei (a keresett számok) az első olyan oldalon, ahol a minta megvan."""
    fajl = forrasok.loc[forrasok["jelentes"] == jelentes, "fajl"].iloc[0]
    for i in range(1, len(pymupdf.open("data/mnb/" + fajl)) + 1):
        talalat = re.search(minta, oldal_szovege(jelentes, i))
        if talalat:
            return talalat.groups()
    return None


def szam_benne(szam, szoveg):
    """Önálló számként szerepel-e a szám a szövegben (a 13,7-ben nincs benne a 3,7). A tizedespont is jó."""
    szoveg = re.sub(r"(\d)\.(\d)", r"\1,\2", szoveg)
    return re.search(rf"(?<![\d,]){re.escape(szam)}(?!\d|,\d)", szoveg) is not None


def alatamasztva(eredmeny, szam):
    """A szám benne van a válaszban, ÉS legalább egy hivatkozott darab nyers PDF-oldalán is."""
    if not szam_benne(szam, eredmeny["valasz"]):
        return False
    for n in hivatkozasok(eredmeny["valasz"]):
        if 1 <= n <= len(eredmeny["talalatok"]):
            d = eredmeny["talalatok"][n - 1][1]
            if szam_benne(szam, oldal_szovege(d["jelentes"], d["pdf_oldal"])):
                return True
    return False


def nem_tudja(valasz):
    """A válasz jelzi, hogy a forrásokban nincs benne, és nem ad meg számot (évszámot kivéve)."""
    jelzi = re.search(r"nincs benne|nem tudom|nem tartalmaz|nem szerepel", valasz.lower()) is not None
    szamok = [s for s in re.findall(r"\d+(?:[,.]\d+)?", valasz) if not re.fullmatch(r"(19|20)\d\d", s)]
    return jelzi and not szamok

In [ ]:
# Az ágensblokk jelentései: az 5-6. alkalom öte, és a 2023. szeptemberi (a mintaprojekt kérdése miatt).
AGENS_JELENTESEK = ["2023-09", "2023-12", "2024-03", "2024-09", "2025-03", "2026-09"]

# A 6. alkalom takarított darabolása (táblázatsorok külön, címkézett mondatként), 6 jelentésre.
def sor_mondat(cellak, evek, cimke):
    """Egy táblázatsorból címkézett mondat. cellak: a sorcím, utána az értékek; evek: a táblázat fejléce."""
    sorcim, ertekek = cellak[0], cellak[1:]
    if len(ertekek) == len(evek):                     # csak akkor párosítjuk az évekkel, ha egyértelmű
        ertekek = [f"{ev}: {ertek}" for ev, ertek in zip(evek, ertekek)]
    eleje = f"{cimke}, {sorcim}" if cimke else sorcim
    return f"{eleje}: " + "; ".join(ertekek)


BETUK = r"[a-zA-ZáéíóöőúüűÁÉÍÓÖŐÚÜŰ]{3}"
SZAMCELLA = r"[\d\s,.\-–−()%]+"            # csak számjegy, vessző, pont, kötőjel, zárójel, százalék


def oldal_bekezdesei_es_sorai(oldal):
    """Az oldal bekezdései (a 3. alkalom szabályával) és a táblázatsorai, címkézett mondatként (sor_mondat)."""
    bekezdesek, sorok = [], []
    cim, csoport, evek = "", "", []
    for blokk in oldal.get_text("blocks"):
        cellak = [c.strip() for c in blokk[4].split("\n") if c.strip()]   # a táblázatban cellánként új sor
        szoveg = tisztit(blokk[4])
        if not cellak or any(jel in szoveg for jel in FEJLEC_JELEK) or szoveg.isupper():
            continue                                        # üres blokk, élőfej, élőláb
        if re.match(r"\d+-\d+\. táblázat:", szoveg):        # táblázatcím: innen új táblázat kezdődik
            cim, csoport, evek = szoveg, "", []
        fejlec = []
        while cellak and re.fullmatch(r"20\d\d", cellak[0]):  # a fejléc évei: 2024, 2025, 2026
            fejlec.append(cellak.pop(0))
        if len(fejlec) >= 2:
            evek = fejlec
            szoveg = tisztit(" ".join(cellak))
        if not cellak:
            continue
        if (len(cellak) >= 2 and re.search(BETUK, cellak[0])
                and all(re.fullmatch(SZAMCELLA, c) for c in cellak[1:])):
            cimke = ", ".join(dict.fromkeys(x for x in [cim, csoport] if x))   # ismétlés nélkül
            sorok.append(sor_mondat(cellak, evek, cimke))  # táblázatsor: szöveg, utána csupa szám
            continue
        if (len(cellak) == 1 and re.search(BETUK, cellak[0]) and not cellak[0].endswith(".")
                and len(cellak[0].split()) <= 10):
            csoport = cellak[0]                             # csoportcím, például: GDP (éves növekedés, %)
        if len(szoveg.split()) >= 5 and re.search(BETUK, szoveg):
            bekezdesek.append(szoveg)                       # a 3. alkalom szabálya
    return bekezdesek, sorok


darabok = []
for s in forrasok[forrasok["jelentes"].isin(AGENS_JELENTESEK)].itertuples():
    for p, oldal_ in enumerate(pymupdf.open("data/mnb/" + s.fajl), start=1):
        bekezdesek, sorok = oldal_bekezdesei_es_sorai(oldal_)
        alap = {"jelentes": s.jelentes, "pdf_oldal": p, "oldal": nyomtatott_oldalszam(oldal_)}
        for i, szoveg in enumerate(darabol_bekezdesenkent(bekezdesek)):
            darabok.append(dict(alap, id=f"{s.jelentes}_p{p:03d}_{i}", szoveg=szoveg))
        for i, szoveg in enumerate(sorok):                  # minden táblázatsor külön darab
            darabok.append(dict(alap, id=f"{s.jelentes}_p{p:03d}_t{i}", szoveg=szoveg))
print(f"{len(darabok)} darab, ebből {sum('_t' in d['id'] for d in darabok)} táblázatsor")

index = index_betoltese(darabok, Path("data/cache/embeddingek_agens.npz"))   # kb. 2-2,5 perc, ha nincs meg

## 2. A mintakérdés, és amit a script tud róla

> A 2023. szeptemberi MNB-jelentés mekkora éves inflációt várt 2023-ra, és mennyi lett?

A várt értékek a nyers PDF-ből és a `data/inflacio_eves.csv`-ből (leírása: `data/inflacio_eves.md`):

In [ ]:
MINTA = "A 2023. szeptemberi MNB-jelentés mekkora éves inflációt várt 2023-ra, és mennyi lett?"

# Az előrejelzés a jelentés mondatából (a nyers PDF-ből)
vart_sav = szam_a_pdfbol(r"Az infláció 2023-ban (\d+,\d) – (\d+,\d) százalék között alakulhat", "2023-09")
print("Az előrejelzés (2023. szeptemberi jelentés):", " – ".join(vart_sav), "százalék")

# Mit jelez előre az MNB? A táblázat címkézett sora (6. alkalom) megmondja.
for d in darabok:
    if d["jelentes"] == "2023-09" and "MNB (2023. szeptember)" in d["szoveg"] and "Fogyasztóiár" in d["szoveg"]:
        print("A jelentés táblázata:", d["szoveg"][d["szoveg"].index("Fogyasztóiár"):], f"({forras_cimke(d)})")

inflacio_tabla = pd.read_csv("data/inflacio_eves.csv", sep=";")
inflacio_tabla[inflacio_tabla["ev"].isin([2023, 2024])]

Melyik oszloppal kell összevetni, és miért? Mi lenne a válasz a másikkal?

## 3. Első szint: chat

Forrás és eszköz nélkül; előbb gondolkodás nélkül, aztán gondolkodással (**kb. 1 perc**).

In [ ]:
RENDSZER_CHAT = "Válaszolj röviden, magyarul, két-három mondatban."

valasz_chat = kerdez(MINTA, rendszer=RENDSZER_CHAT, gondolkodas="none")
print(valasz_chat)

kezdes = time.perf_counter()
valasz_gondolkodva = kerdez(MINTA, rendszer=RENDSZER_CHAT)
print(f"\nGondolkodással ({time.perf_counter() - kezdes:.0f} mp, {utolso_hasznalat['completion_tokens']} token):")
print(valasz_gondolkodva)

Szerepelnek a válaszokban a várt számok?

In [ ]:
ksh_2023 = inflacio_tabla.loc[inflacio_tabla["ev"] == 2023, "ksh_fogyasztoiar"].iloc[0]
vart_szamok = {"az előrejelzés alja": vart_sav[0], "az előrejelzés teteje": vart_sav[1],
               "a tényadat (KSH)": f"{ksh_2023}".replace(".", ",")}
for nev, valasz in [("gondolkodás nélkül", valasz_chat), ("gondolkodással", valasz_gondolkodva)]:
    print(nev + ":", {k: szam_benne(v, valasz) for k, v in vart_szamok.items()})

Honnan vehetné a modell ezeket a számokat? Segített a gondolkodás?

## 4. Második szint: munkafolyamat

A lépéseket mi döntjük el: (1) keresés a megnevezett jelentésben, (2) a modell kiolvassa a sávot JSON-ben, (3) tényadat a KSH táblájából, (4) összevetés Pythonban.

### Feladat 4.1 – Copilottal

> Írj egy `tenyadat(ev)` függvényt, amely a `data/inflacio_eves.csv` fájlból (pontosvessző az elválasztó) visszaadja az adott év éves inflációját, tört számként.

Olvasd el, mielőtt **Keep**-et nyomsz: melyik oszlopot választotta a Copilot, és miért? Megnézte a `data/inflacio_eves.md`-t? Mit mondott a 2. rész a jelentés táblázatáról?

In [ ]:
# Ide kerül a Copilot kódja

In [ ]:
# A várt érték: a KSH fogyasztóiár-indexe, mert az MNB ezt jelzi előre (2. rész).
_tabla = pd.read_csv("data/inflacio_eves.csv", sep=";").set_index("ev")
ellenoriz_fuggveny("tenyadat", [((ev,), float(_tabla.loc[ev, "ksh_fogyasztoiar"])) for ev in [2024, 2023, 2022]])

Az én változatom, és a munkafolyamat többi része (olvasd el a megjegyzéseket):

In [ ]:
def tenyadat(ev):
    """Az éves átlagos infláció (%) a KSH fogyasztóiár-indexéből, a data/inflacio_eves.csv-ből."""
    tabla = pd.read_csv("data/inflacio_eves.csv", sep=";")
    return float(tabla.loc[tabla["ev"] == ev, "ksh_fogyasztoiar"].iloc[0])

In [ ]:
def keres_jelentesben(kerdes, jelentes, k=5):
    """Keresés csak egy jelentés darabjai között: ha a kérdés megnevezi a jelentést, a többi csak zavar."""
    sorok = [i for i, d in enumerate(darabok) if d["jelentes"] == jelentes]
    return keres(kerdes, [darabok[i] for i in sorok], index[sorok], k)


def json_bol(szoveg):
    """A modell válaszából a JSON (a ```json keret nélkül); ha nem olvasható, None."""
    szoveg = re.sub(r"^```(json)?|```$", "", szoveg.strip()).strip()
    try:
        return json.loads(szoveg)
    except json.JSONDecodeError:
        return None


KIOLVASO = """Az MNB Inflációs jelentéseinek részleteiből dolgozol. A forrásokból add meg a kérdezett előrejelzést: a sáv alsó és felső szélét, százalékban, és a forrás sorszámát.
Csak ezt a JSON-t írd, más szöveget ne: {"also": 0.0, "felso": 0.0, "forras": 1}
Ha a források nem tartalmazzák, ezt írd: {"also": null, "felso": null, "forras": null}"""


def munkafolyamat(kerdes, jelentes, ev):
    """Előre rögzített lépések: keresés a jelentésben → a modell kiolvassa a sávot → tényadat → összevetés."""
    # 1. lépés: keresés, csak a megnevezett jelentésben
    talalatok = keres_jelentesben(kerdes, jelentes)
    # 2. lépés: a modell kiolvassa az előrejelzés sávját (JSON-ben, hogy a kód tovább tudja vinni)
    uzenetek = [{"role": "system", "content": KIOLVASO},
                {"role": "user", "content": f"Források:\n\n{kontextus(talalatok)}\n\nKérdés: {kerdes}"}]
    sav = json_bol(chat(uzenetek, gondolkodas="none")["content"]) or {}
    print(f"2. lépés, a modell kiolvasta ({utolso_hasznalat['prompt_tokens']} token): {sav}")
    if sav.get("also") is None or not sav.get("forras"):
        print("A modell nem talált előrejelzést: a munkafolyamat itt megáll.")
        return None
    forras = forras_cimke(talalatok[sav["forras"] - 1][1])
    # 3. lépés: a tényadat, mindig ugyanabból a táblából, ugyanabból az oszlopból
    teny = tenyadat(ev)
    # 4. lépés: az összevetés Pythonban
    if teny < sav["also"]:
        helyzet = "a sáv alatt"
    elif teny > sav["felso"]:
        helyzet = "a sáv fölött"
    else:
        helyzet = "a sávon belül"
    print(f"Előrejelzés: {sav['also']}–{sav['felso']}% ({forras}). Tényadat ({ev}, KSH): {teny}%, {helyzet}.")
    return {"also": sav["also"], "felso": sav["felso"], "forras": forras, "teny": teny, "helyzet": helyzet}

In [ ]:
eredmeny_minta = munkafolyamat("Mekkora éves átlagos inflációt várt az MNB 2023 egészére?", "2023-09", 2023)

In [ ]:
_also, _felso = (float(s.replace(",", ".")) for s in vart_sav)
ellenoriz_feltetel("eredmeny_minta",
                   lambda e: e is not None and (e["also"], e["felso"]) == (_also, _felso)
                             and e["teny"] == ksh_2023 and e["helyzet"] == "a sávon belül",
                   f"a sáv {_also}–{_felso} (a PDF-ből), a tényadat {ksh_2023} (KSH), a sávon belül")

### Feladat 4.2 – olvasás

Ugyanez a munkafolyamat ezzel a kérdéssel: „Mekkora GDP-növekedést várt az MNB 2023 egészére?” Mit csinál? `"a"`: hibát dob; `"b"`: a GDP tényadatával veti össze; `"c"`: az inflációval veti össze, hiba nélkül.

In [ ]:
tipp_4_2 = None  # "a", "b" vagy "c"

In [ ]:
eredmeny_gdp = munkafolyamat("Mekkora GDP-növekedést várt az MNB 2023 egészére?", "2023-09", 2023)
ellenoriz("tipp_4_2", vart="c")

Miért nem vette észre a munkafolyamat, hogy rossz adatot kér le? Mikor jobb mégis a munkafolyamat?

## 5. Harmadik szint: ágens (előzetes)

Ma a modell csak **tervez**: megkapja az eszközök listáját, és JSON-ben leírja a lépéseket (gondolat, eszköz, bemenet). Végrehajtás nincs: az a 8–9. alkalom.

In [ ]:
TERVEZO = """Egy ágens tervezője vagy. Az ágens ezeket az eszközöket használhatja:
- dokumentum_kereses(kerdes, jelentes): részleteket keres az MNB Inflációs jelentéseiben; a jelentes formája "ÉÉÉÉ-HH"
- inflacio(ev, forras): Magyarország éves inflációja; forras: "KSH" vagy "Eurostat"
- gdp_novekedes(ev): Magyarország GDP-növekedése az adott évben, %
- szamol(kifejezes): kiszámol egy matematikai kifejezést

Ne válaszolj a kérdésre. Írd le, milyen lépésekben oldaná meg az ágens. Csak egy JSON-listát írj, más szöveget ne:
[{"gondolat": "miért ez a lépés", "eszkoz": "az eszköz neve", "bemenet": {"parameter": "érték"}}]
Ha olyan eszköz kellene, ami nincs a listán, akkor is írd bele a lépést, a kitalált eszköz nevével."""

ESZKOZ_NEVEK = ["dokumentum_kereses", "inflacio", "gdp_novekedes", "szamol"]


def terv(feladat):
    """A modell terve a feladathoz: lépések listája (gondolat, eszköz, bemenet). Csak terv, nem futtat semmit."""
    lepesek = json_bol(kerdez(feladat, rendszer=TERVEZO, gondolkodas="none")) or []
    for i, l in enumerate(lepesek, start=1):
        jel = "" if l.get("eszkoz") in ESZKOZ_NEVEK else "   <-- NINCS ILYEN ESZKÖZÜNK"
        print(f"{i}. {l.get('eszkoz')}({json.dumps(l.get('bemenet'), ensure_ascii=False)}){jel}")
        print(f"   gondolat: {l.get('gondolat')}")
    return lepesek


def hasznalt_eszkozok(lepesek):
    """A tervben szereplő eszközök nevei, ismétlés nélkül."""
    return sorted({l.get("eszkoz") for l in lepesek})

In [ ]:
terv_minta = terv(MINTA)

És a GDP-kérdéssel?

In [ ]:
terv_gdp = terv("A 2023. szeptemberi MNB-jelentés mekkora GDP-növekedést várt 2023-ra, és mennyi lett?")

In [ ]:
ellenoriz_feltetel("terv_gdp", lambda t: "gdp_novekedes" in hasznalt_eszkozok(t) and "inflacio" not in hasznalt_eszkozok(t),
                   "a GDP-kérdés tervében a gdp_novekedes eszköz szerepel, az inflacio nem")
ellenoriz_feltetel("terv_minta", lambda t: "dokumentum_kereses" in hasznalt_eszkozok(t) and "inflacio" in hasznalt_eszkozok(t),
                   "a mintakérdés tervében keresés a jelentésben és inflációs adat szerepel")

Olvasd el a mintakérdés tervét: mi hiányzik belőle, vagy mi felesleges? Ki ellenőrzi a tervet?

### Feladat 5.1 – a te többlépéses feladatod

Írj egy feladatot, amelyet egy asszisztens több lépésben oldana meg (például: „Hasonlítsd össze az inflációt a négy visegrádi országban az utóbbi öt évben, és mondd meg, hol volt a legmagasabb.”), vagy a projektedhez kapcsolódót. Mit tervez a modell, és milyen eszköz hiányzik hozzá? A hiányzó eszközt a script jelöli (`<-- NINCS ILYEN ESZKÖZÜNK`).

In [ ]:
sajat_feladat = None  # a None helyére írd a feladatot, idézőjelek közé

In [ ]:
ellenoriz_feltetel("sajat_feladat", lambda s: isinstance(s, str) and len(s.split()) >= 8,
                   "egy többlépéses feladat, legalább egy teljes mondatban (8 szó)")
if sajat_feladat:
    sajat_terv = terv(sajat_feladat)
    print("\nHiányzó eszközök:", [e for e in hasznalt_eszkozok(sajat_terv) if e not in ESZKOZ_NEVEK] or "nincs")

Jók a lépések és a sorrend? Ami hiányzik, honnan lehetne adatot kérni hozzá?

## 6. A projektben

### Otthonra

Írj egy `agens_terv.md` fájlt a projekted repójába (párban: közösen):

1. a projekted fő kérdése (a mintában: mit várt a 2023. szeptemberi jelentés 2023-ra, és mennyi lett);
2. a lépések, ahogy egy **munkafolyamat** végezné (mint a 4. részben); jelöld, melyik lépésnél kellene döntés, ha a kérdés változik;
3. az **eszközök** listája: név, mit csinál (egy mondat), bemenetei (névvel és formával, például `ev`: egész szám), kimenete. A csomagod adatlekérő függvénye (`adatlekeres.py`) az első eszközöd;
4. melyik adatsorral kell összevetni a dokumentum állításait, és miből derül ki (oldalszámmal). A mintában: a KSH fogyasztóiár-indexével, mert a jelentés táblázata (26. oldal) a fogyasztóiár-indexet jelzi előre.

A 8. alkalommal ebből lesz az eszközök JSON-leírása.